# Chapter 05: Multifingered Hand Kinematics

Source orientation: printed pp. 211-264; PDF pp. 229-282. Chapter question: **How do local contact forces become object-level motion and wrench capability?**

This notebook is an original, standalone teaching chapter. It uses the textbook for structure and mathematical orientation, but the explanations, code, and figures are created here. The chapter focus is contacts, grasp maps, force closure, grasp planning, rolling contact. The key objects are contact frames, friction cones, contact wrench bases, grasp maps, convex hulls, internal forces, closure margins, rolling coordinates, and hand Jacobians.

Generated artifacts live under `artifacts/chapter-05` and are displayed both by code and in the static gallery. The final sanity cell checks the mathematical claims and artifact files so that the notebook remains auditable after reruns.


## Translation Guide

The chapter converts local contact forces into an object-level wrench map. Force closure becomes a convex geometry question: can positive combinations of contact wrench directions balance every small disturbance? The computational translation used here is deliberately concrete: every named object becomes an array, graph, cone, trajectory, or map whose dimensions can be checked. That keeps the notebook independent from the PDF while preserving the chapter's mathematical route.

The main objects for this chapter are contact frames, friction cones, contact wrench bases, grasp maps, convex hulls, internal forces, closure margins, rolling coordinates, and hand Jacobians. Read each one as a map between spaces. Ask what frame or chart supplies its coordinates, what operation changes those coordinates, and what quantity should remain invariant. The visuals in this notebook are chosen to make those questions inspectable rather than decorative.

The source pages are used only as orientation. Definitions and examples are paraphrased into course language, and all figures are generated from fresh code. When a visual resembles a textbook concept, it is a redraw or computational experiment rather than a page crop.


## Route Through The Chapter

We move in four passes. First, the notebook names the chapter's geometric objects and their engineering purpose. Second, it generates the visual sequence: contact model gallery, grasp map wrench space, force closure convex test. Third, it runs a compact computational check that records the relevant residuals, ranks, endpoint errors, determinants, or signs. Fourth, it closes with an applied lab that asks the reader to change a parameter and explain what stayed invariant.

The important habit is to compare the visual artifact with the JSON check. If a cone claims feasibility, the feasibility check should agree. If a Jacobian claims usable motion, the task-space determinant or rank should agree. If a loop claims bracket displacement, the endpoint check should reveal it. The notebook is designed so those cross-checks are near each other.


In [ ]:
from pathlib import Path
import sys
import numpy as np

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the robotics course root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts"
TOPIC = "chapter-05"

from utils.artifacts import display_artifact, save_json
from utils.visuals import build_storyboard, storyboard_check_payload


In [ ]:
storyboard = {
  "label": "Chapter 05: Multifingered Hand Kinematics",
  "artifact_topic": "chapter-05",
  "source_span": "printed pp. 211-264; PDF pp. 229-282",
  "visual_sequence": [
    {
      "kind": "cone",
      "concept": "Contact Model Gallery",
      "filename": "contact-model-gallery.png",
      "observation": "friction cones and contact wrench directions"
    },
    {
      "kind": "grasp",
      "concept": "Grasp Map Wrench Space",
      "filename": "grasp-map-wrench-space.png",
      "observation": "contact columns summing to object wrench"
    },
    {
      "kind": "closure",
      "concept": "Force Closure Convex Test",
      "filename": "force-closure-convex-test.png",
      "observation": "origin containment in wrench space"
    }
  ]
}

visual_results = build_storyboard(storyboard, ARTIFACT_ROOT, TOPIC)
visual_payload = storyboard_check_payload(storyboard, visual_results)
for item in visual_results:
    display_artifact(item["path"], width=720)
visual_payload


## Static Artifact Gallery

![Contact Model Gallery](../../artifacts/chapter-05/figures/contact-model-gallery.png)

*Inspection target:* friction cones and contact wrench directions.

![Grasp Map Wrench Space](../../artifacts/chapter-05/figures/grasp-map-wrench-space.png)

*Inspection target:* contact columns summing to object wrench.

![Force Closure Convex Test](../../artifacts/chapter-05/figures/force-closure-convex-test.png)

*Inspection target:* origin containment in wrench space.


## Worked Interpretation

The grasp computation builds planar contact wrench columns from points, normals, and friction edges, then tests whether the origin lies inside their convex hull. The plotted hull is the visual version of the linear feasibility check. This is the chapter's small worked example, not a full simulator. It is intentionally narrow enough that the relevant convention can be seen directly in code and broad enough to catch the common failure mode.

The visual sequence and the executable check use compatible parameters whenever possible. The point is to avoid a split course where the images tell one story and the numbers tell another. When extending this notebook, reuse that pattern: pick one invariant, draw the geometry that exposes it, then save a check payload that would fail if the convention were reversed or the rank condition were lost.

**Pitfall to watch:** More fingers do not automatically mean force closure. Contact normal orientation, friction coefficient, contact placement, and soft-finger assumptions decide which wrench directions are actually available. This pitfall is why the notebook avoids silent coordinate conventions. Names, frames, dimensions, and signs are part of the teaching surface, not implementation trivia.


In [ ]:
from utils.grasping import grasp_wrenches, internal_force_basis, origin_in_convex_hull

points = [np.array([-0.5, 0.0]), np.array([0.5, 0.0]), np.array([0.0, 0.55])]
normals = [np.array([1.0, 0.15]), np.array([-1.0, 0.15]), np.array([0.0, -1.0])]
G = grasp_wrenches(points, normals, mu=0.8)
N = internal_force_basis(G)
check_payload = {
    "grasp_map_shape": list(G.shape),
    "grasp_rank": int(np.linalg.matrix_rank(G)),
    "origin_in_convex_hull": origin_in_convex_hull(G),
    "internal_force_dimension": int(N.shape[1]),
    "internal_force_residual": float(np.linalg.norm(G @ N)) if N.size else 0.0,
}
assert check_payload["grasp_rank"] == 3
assert check_payload["origin_in_convex_hull"]
check_payload


## Applied Lab

Lab prompt: Move frictional contacts around a polygon and watch the closure margin change.

Before running the lab, make a prediction in three parts: which geometric object is being changed, which representation will show the change most clearly, and which scalar check should move in the same direction. After running it, compare the prediction with the saved JSON payload under `artifacts/chapter-05/labs`.

Use the pitfall above as a diagnostic. If the visual and scalar check disagree, inspect the frame convention, normalization, rank threshold, sign convention, or parameter range. The best result is not merely a green check; it is a short explanation of why the check protects the chapter's main idea.


In [ ]:
lab_notes = {
    "lab": 'Move frictional contacts around a polygon and watch the closure margin change.',
    "source_orientation": "printed pp. 211-264; PDF pp. 229-282",
    "artifact_topic": TOPIC,
    "reader_task": "Change one parameter, rerun the visual cell, and explain which invariant did or did not change.",
}
lab_path = save_json(lab_notes, TOPIC, "labs", "applied-lab.json", root=ARTIFACT_ROOT)
display_artifact(lab_path)


In [ ]:
from pathlib import Path

visual_paths = [Path(item["path"]) for item in visual_results]
assert visual_payload["assertions"]["has_multiple_visuals"]
assert visual_payload["assertions"]["all_visuals_nonblank"]
assert all(path.exists() and path.stat().st_size > 1000 for path in visual_paths)

final_sanity = {
    "visual_payload": visual_payload,
    "checks": check_payload,
    "visual_artifact_count": len(visual_paths),
    "visual_artifacts": [path.relative_to(BOOK_ROOT).as_posix() for path in visual_paths],
}
sanity_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json", root=ARTIFACT_ROOT)
display_artifact(sanity_path)
final_sanity


## Practice And Inspection Notes

Use this chapter as a small laboratory, not as a static summary. The source span is printed pp. 211-264 and PDF pp. 229-282, but the working material is the notebook itself: the generated artifacts, the executable check payload, and the applied lab. Start by reading the chapter question again: **How do local contact forces become object-level motion and wrench capability?** Then identify which part of the visual sequence gives the most direct answer. For this chapter the focus is contacts, grasp maps, force closure, grasp planning, rolling contact, so the useful inspection targets are not generic screenshots; they are the ranks, cones, motions, frames, phase loops, energy shapes, or dependency arrows that make that focus concrete.

The `contact model gallery` artifact uses the `cone` visual family; inspect it for friction cones and contact wrench directions. The `grasp map wrench space` artifact uses the `grasp` visual family; inspect it for contact columns summing to object wrench. The `force closure convex test` artifact uses the `closure` visual family; inspect it for origin containment in wrench space.

After viewing the artifacts, rerun the computational check cell and read the keys in `check_payload` as a miniature rubric. Each key should protect one concept from a plausible robotics mistake. A determinant or rank protects a singularity claim. A residual protects an equation or closure condition. A monotonicity flag protects a scale-law interpretation. An endpoint error protects a steering construction. A power-invariance error protects a frame transformation. If a check is near zero, ask why zero is the right target; if a check is positive, ask what physical or geometric margin it measures.

Try three variations. First, make a small parameter change that should preserve the chapter's main invariant, such as a coordinate-frame change, a harmless redraw scale, or a sample density increase. Second, make a parameter change that should stress the model, such as a near-singular joint pose, lower friction coefficient, larger controller delay, smaller bracket loop, or weaker tendon tension. Third, make a convention change deliberately, such as reversing a normal or swapping a body/spatial interpretation, and predict which check should fail. These variations turn the notebook from a solved example into a diagnostic tool.

When writing your own notes, use this sentence pattern: "The artifact shows ..., the computation checks ..., and the invariant is ... ." That pattern is intentionally repetitive because it catches vague understanding. A reader who can fill in those three blanks for this chapter can usually transfer the idea to a different mechanism, contact model, or control task without reopening the textbook.


## Takeaways

- A grasp map is the bridge from local contact forces to object wrench space.
- Force closure is best inspected as convex containment plus a numerical margin.
- Rolling contact changes contact coordinates, so grasp capability can evolve even when the object shape stays fixed.
- Revisit the saved artifacts after changing parameters; the strongest learning usually comes from explaining why the invariant survived.
